# GPT-Based ASR Transcript Scoring

Uses **OpenAI GPT-5.4** to evaluate how accurately each ASR model transcribed
compared to the original transcript.

For every row in each language sheet, GPT scores each model transcript on a
**0-100** scale across three dimensions:
- **Accuracy** - factual/content correctness
- **Fluency** - grammar, readability, natural flow
- **Completeness** - nothing omitted or hallucinated

A weighted **Overall Score** is also computed.

In [1]:
# ============================================================
# IMPORTS + CONFIG
# ============================================================
import os
import json
import time
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

# Load API key from .env
load_dotenv()
client = OpenAI()  # picks up OPENAI_API_KEY from environment

# ============================================================
# CONFIGURATION
# ============================================================
INPUT_FILE  = r"D:\audio_transscript\gemnai\MCF Languages.xlsx"
OUTPUT_FILE = r"D:\audio_transscript\gemnai\MCF Languages_with_scores.xlsx"

ORIGINAL_COL = "Original_transcript"

MODEL_COLS = {
    "OmniASR":    "OmniASR_transcript-7B",
    "ElevenLabs": "ElevenLabs - Scribe V2",
    "Gemini":     "Gemini -2.5- Pro",
}

SHEET_NAMES = ["Hausa", "Igbo", "Yoruba", "Pidgin "]

GPT_MODEL = "gpt-5.4"   # latest flagship model

print("OpenAI client initialised")
print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")
print(f"Model:  {GPT_MODEL}")
print(f"Sheets: {SHEET_NAMES}")

OpenAI client initialised
Input:  D:\audio_transscript\gemnai\MCF Languages.xlsx
Output: D:\audio_transscript\gemnai\MCF Languages_with_scores.xlsx
Model:  gpt-5.4
Sheets: ['Hausa', 'Igbo', 'Yoruba', 'Pidgin ']


In [2]:
# ============================================================
# GPT SCORING PROMPT + FUNCTION
# ============================================================

SYSTEM_PROMPT = """You are an expert linguist and ASR (Automatic Speech Recognition) evaluator.
You will be given an ORIGINAL transcript and a MODEL transcript produced by an ASR system.
Your job is to score how well the MODEL transcript matches the ORIGINAL transcript.

Score on these three dimensions (0-100 each):

1. **Accuracy** (0-100): How factually and content-wise correct is the model transcript
   compared to the original? Are the words, names, numbers, and meaning preserved?

2. **Fluency** (0-100): How grammatically correct, readable, and natural is the model
   transcript? Does it flow well as natural speech/text?

3. **Completeness** (0-100): Is the model transcript complete? Does it capture all the
   content from the original without omissions or hallucinated additions?

IMPORTANT RULES:
- Compare the MODEL transcript against the ORIGINAL transcript only.
- Be fair and consistent in your scoring.
- Minor spelling/punctuation differences should not heavily penalise scores.
- If either transcript is empty or nonsensical, give scores of 0.

Respond with ONLY a JSON object in this exact format (no markdown, no extra text):
{"accuracy": <int>, "fluency": <int>, "completeness": <int>}
"""


def build_user_prompt(original, model_transcript):
    return (
        f"ORIGINAL TRANSCRIPT:\n{original}\n\n"
        f"MODEL TRANSCRIPT:\n{model_transcript}"
    )


def score_transcript(original, model_transcript, max_retries=3):
    """
    Call GPT to score a single model transcript against the original.
    Returns dict with keys: accuracy, fluency, completeness, overall
    """
    if pd.isna(original) or pd.isna(model_transcript):
        return {"accuracy": np.nan, "fluency": np.nan, "completeness": np.nan, "overall": np.nan}

    original = str(original).strip()
    model_transcript = str(model_transcript).strip()

    if not original or not model_transcript:
        return {"accuracy": np.nan, "fluency": np.nan, "completeness": np.nan, "overall": np.nan}

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": build_user_prompt(original, model_transcript)},
                ],
                temperature=0.0,
                max_completion_tokens=100,
            )

            raw = response.choices[0].message.content.strip()

            # Parse JSON - handle possible markdown wrapping
            if raw.startswith("```"):
                raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

            scores = json.loads(raw)

            accuracy     = int(scores.get("accuracy", 0))
            fluency      = int(scores.get("fluency", 0))
            completeness = int(scores.get("completeness", 0))

            # Weighted overall: Accuracy 40%, Completeness 35%, Fluency 25%
            overall = round(accuracy * 0.40 + completeness * 0.35 + fluency * 0.25, 2)

            return {
                "accuracy": accuracy,
                "fluency": fluency,
                "completeness": completeness,
                "overall": overall,
            }

        except json.JSONDecodeError:
            print(f"      JSON parse error (attempt {attempt+1}), retrying...")
            time.sleep(1)
        except Exception as e:
            print(f"      API error (attempt {attempt+1}): {e}")
            time.sleep(2 ** attempt)

    # All retries failed
    return {"accuracy": np.nan, "fluency": np.nan, "completeness": np.nan, "overall": np.nan}

print("Scoring function ready")

Scoring function ready


In [3]:
# ============================================================
# PROCESS ALL SHEETS
# ============================================================

all_sheets_data = {}
summary_rows = []

for sheet_name in SHEET_NAMES:
    print("=" * 60)
    print(f"Processing sheet: {sheet_name.strip()}")
    print("=" * 60)

    df = pd.read_excel(INPUT_FILE, sheet_name=sheet_name)
    # Drop unnamed columns
    df = df[[c for c in df.columns if c is not None and "Unnamed" not in str(c)]]

    originals = df[ORIGINAL_COL].tolist()
    n_rows = len(originals)
    print(f"  Total rows: {n_rows}")

    # Find available models in this sheet
    available_models = {}
    for model_name, col_name in MODEL_COLS.items():
        if col_name in df.columns:
            non_null = df[col_name].notna().sum()
            if non_null > 0:
                available_models[model_name] = col_name
                print(f"  OK {model_name}: {non_null} transcripts")
            else:
                print(f"  WARN {model_name}: column exists but no data")
        else:
            print(f"  SKIP {model_name}: column not found")

    # Score each model
    model_scores_summary = {}

    for model_name, col_name in available_models.items():
        print(f"\n  Scoring {model_name} with {GPT_MODEL}...")

        accuracy_scores = []
        fluency_scores = []
        completeness_scores = []
        overall_scores = []

        for i in range(n_rows):
            orig = originals[i]
            trans = df[col_name].iloc[i]

            result = score_transcript(orig, trans)
            accuracy_scores.append(result["accuracy"])
            fluency_scores.append(result["fluency"])
            completeness_scores.append(result["completeness"])
            overall_scores.append(result["overall"])

            # Progress indicator every 5 rows
            if (i + 1) % 5 == 0 or i == n_rows - 1:
                print(f"    Scored {i+1}/{n_rows} rows")

            # Rate limiting: small delay between API calls
            time.sleep(0.3)

        print(f"    Scored {n_rows}/{n_rows} rows - done")

        # Add score columns to dataframe
        df[f"{model_name}_GPT_Accuracy"]     = accuracy_scores
        df[f"{model_name}_GPT_Fluency"]      = fluency_scores
        df[f"{model_name}_GPT_Completeness"] = completeness_scores
        df[f"{model_name}_GPT_Overall"]      = overall_scores

        # Compute averages for summary (ignore NaN)
        valid_acc = [s for s in accuracy_scores if not (isinstance(s, float) and np.isnan(s))]
        valid_flu = [s for s in fluency_scores if not (isinstance(s, float) and np.isnan(s))]
        valid_comp = [s for s in completeness_scores if not (isinstance(s, float) and np.isnan(s))]
        valid_overall = [s for s in overall_scores if not (isinstance(s, float) and np.isnan(s))]

        avg_acc  = np.mean(valid_acc) if valid_acc else 0.0
        avg_flu  = np.mean(valid_flu) if valid_flu else 0.0
        avg_comp = np.mean(valid_comp) if valid_comp else 0.0
        avg_ovr  = np.mean(valid_overall) if valid_overall else 0.0

        model_scores_summary[model_name] = {
            "accuracy": avg_acc,
            "fluency": avg_flu,
            "completeness": avg_comp,
            "overall": avg_ovr,
        }

        print(f"    Avg Accuracy:     {avg_acc:.2f}")
        print(f"    Avg Fluency:      {avg_flu:.2f}")
        print(f"    Avg Completeness: {avg_comp:.2f}")
        print(f"    Avg Overall:      {avg_ovr:.2f}")

    # --- Best model per row (only for rows that have scores) ---
    overall_cols = [f"{m}_GPT_Overall" for m in available_models if f"{m}_GPT_Overall" in df.columns]
    if overall_cols:
        best_models = []
        best_scores = []
        for idx, row in df.iterrows():
            vals = {c: row[c] for c in overall_cols if pd.notna(row[c])}
            if vals:
                best_col = max(vals, key=vals.get)
                best_models.append(best_col.replace("_GPT_Overall", ""))
                best_scores.append(vals[best_col])
            else:
                best_models.append("")
                best_scores.append(np.nan)
        df["GPT_Best_Model"] = best_models
        df["GPT_Best_Score"] = best_scores

    # --- Summary rows ---
    for model_name, scores in model_scores_summary.items():
        summary_rows.append({
            "Language": sheet_name.strip(),
            "Model": model_name,
            "GPT_Avg_Accuracy": round(scores["accuracy"], 2),
            "GPT_Avg_Fluency": round(scores["fluency"], 2),
            "GPT_Avg_Completeness": round(scores["completeness"], 2),
            "GPT_Avg_Overall": round(scores["overall"], 2),
        })

    all_sheets_data[sheet_name] = df
    print(f"\n  {sheet_name.strip()} done!\n")

Processing sheet: Hausa
  Total rows: 5
  OK OmniASR: 3 transcripts
  OK ElevenLabs: 3 transcripts
  OK Gemini: 3 transcripts

  Scoring OmniASR with gpt-5.4...
    Scored 5/5 rows
    Scored 5/5 rows - done
    Avg Accuracy:     81.33
    Avg Fluency:      78.00
    Avg Completeness: 79.33
    Avg Overall:      79.80

  Scoring ElevenLabs with gpt-5.4...
    Scored 5/5 rows
    Scored 5/5 rows - done
    Avg Accuracy:     75.67
    Avg Fluency:      76.67
    Avg Completeness: 88.33
    Avg Overall:      80.35

  Scoring Gemini with gpt-5.4...
    Scored 5/5 rows
    Scored 5/5 rows - done
    Avg Accuracy:     87.33
    Avg Fluency:      84.67
    Avg Completeness: 93.00
    Avg Overall:      88.65

  Hausa done!

Processing sheet: Igbo
  Total rows: 3
  OK OmniASR: 3 transcripts
  OK ElevenLabs: 3 transcripts
  OK Gemini: 3 transcripts

  Scoring OmniASR with gpt-5.4...
    Scored 3/3 rows
    Scored 3/3 rows - done
    Avg Accuracy:     62.00
    Avg Fluency:      38.00
    Avg Com

In [4]:
# ============================================================
# BUILD GPT SCORE SUMMARY
# ============================================================

summary_df = pd.DataFrame(summary_rows)
print("\nGPT Score Summary (all languages):\n")
print(summary_df.to_string(index=False))

# Best model per language
print("\n" + "=" * 60)
print("BEST MODEL PER LANGUAGE (by GPT Overall Score):")
print("=" * 60)

best_rows = []
for lang in summary_df["Language"].unique():
    lang_data = summary_df[summary_df["Language"] == lang]
    if lang_data["GPT_Avg_Overall"].max() == 0:
        print(f"  {lang:10s} -> No valid scores")
        continue
    best = lang_data.loc[lang_data["GPT_Avg_Overall"].idxmax()]
    best_rows.append({
        "Language": lang,
        "Best_Model": best["Model"],
        "GPT_Avg_Accuracy": best["GPT_Avg_Accuracy"],
        "GPT_Avg_Fluency": best["GPT_Avg_Fluency"],
        "GPT_Avg_Completeness": best["GPT_Avg_Completeness"],
        "GPT_Avg_Overall": best["GPT_Avg_Overall"],
    })
    print(f"  {lang:10s} -> {best['Model']} (Overall: {best['GPT_Avg_Overall']:.2f})")

best_df = pd.DataFrame(best_rows)
print("\nSummary tables built")


GPT Score Summary (all languages):

Language      Model  GPT_Avg_Accuracy  GPT_Avg_Fluency  GPT_Avg_Completeness  GPT_Avg_Overall
   Hausa    OmniASR             81.33            78.00                 79.33            79.80
   Hausa ElevenLabs             75.67            76.67                 88.33            80.35
   Hausa     Gemini             87.33            84.67                 93.00            88.65
    Igbo    OmniASR             62.00            38.00                 79.67            62.18
    Igbo ElevenLabs             30.00            56.00                 33.67            37.78
    Igbo     Gemini             49.67            58.00                 73.33            60.03
  Yoruba    OmniASR             33.00            32.00                 44.67            36.83
  Yoruba ElevenLabs             36.33            59.33                 53.33            48.03
  Yoruba     Gemini             36.67            70.67                 52.67            50.77
  Pidgin    OmniASR    

In [5]:
# ============================================================
# SAVE TO EXCEL
# ============================================================

print("Saving results to Excel...")

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    for sheet_name in SHEET_NAMES:
        df = all_sheets_data[sheet_name]
        df.to_excel(writer, sheet_name=sheet_name.strip(), index=False)
        print(f"  Saved: {sheet_name.strip()} ({len(df)} rows, {len(df.columns)} cols)")

    # GPT scores summary sheet
    summary_df.to_excel(writer, sheet_name="GPT Score Summary", index=False)
    print(f"  Saved: GPT Score Summary")

    # Best model per language sheet
    best_df.to_excel(writer, sheet_name="GPT Best Model", index=False)
    print(f"  Saved: GPT Best Model")

print(f"\nDONE! Output saved to:\n   {OUTPUT_FILE}")

Saving results to Excel...
  Saved: Hausa (5 rows, 23 cols)
  Saved: Igbo (3 rows, 23 cols)
  Saved: Yoruba (3 rows, 23 cols)
  Saved: Pidgin (24 rows, 23 cols)
  Saved: GPT Score Summary
  Saved: GPT Best Model

DONE! Output saved to:
   D:\audio_transscript\gemnai\MCF Languages_with_scores.xlsx
